# 數位控制系統第六章：系統時間響應特性（教學版 Notebook）

本 Notebook 是 `chp6.md` 教材的教學版，額外補充：

- 每個第一次出現的 MATLAB / Octave 函數（`residue`, `deconv`, `polyval`, `roots`, `single`, ...）的逐步解說
- `log` vs `log10`、`angle` 回傳弧度、差分方程式更新順序等初學者容易卡住的地方
- 公式 → 程式碼的逐項對照（每段程式都標註對應的教材式號與範例編號）
- 七個可直接執行的實驗，親手驗證本章每一條公式

建議搭配 `chp6.md`（完整理論、推導與符號定義）與 `chp6.m`（精簡可執行版）一起閱讀。

> **本章要回答的問題**：第 5 章算出了閉迴路 $C(z)$。**知道了 $C(z)$，系統實際上會怎麼動？** 包括暫態（多快、震不震盪）與穩態（最後停在哪、誤差多大）。

---


## 🔧 環境設定

> **需要 Control System Toolbox**（Octave 為 `control` 套件）。


In [ ]:
%plot --format svg

if exist('OCTAVE_VERSION', 'builtin')
    warning('off', 'Octave:gnuplot-graphics');
    warning('off', 'Octave:fltk-graphics');
    graphics_toolkit('gnuplot');
    pkg load control;
end
clear; clc;

set(0, 'DefaultTextFontName', 'Microsoft JhengHei');
set(0, 'DefaultAxesFontName', 'Microsoft JhengHei');

printf('環境就緒\n');

## 📖 全章符號總表

| 符號 | 意義 |
|---|---|
| $T(z)$ | 閉迴路轉移函數 $C(z)/R(z)$ |
| $z_1=r\angle\pm\theta$ | $z$ 平面的一對共軛極點 |
| $r$ | 極點的**絕對值** → 決定**衰減速度** |
| $\theta$ | 極點的**輻角（弧度）** → 決定**振盪速度** |
| $\zeta$ | **阻尼比** |
| $\omega_n$ | **自然頻率** |
| $\tau$ | **時間常數** |
| $N$ | **系統型式**：$G(z)$ 在 $z=1$ 的極點數 |
| $K_P$ | **位置誤差常數** $=\lim_{z\to1}G(z)$ |
| $K_v$ | **速度誤差常數** $=\lim_{z\to1}(z-1)G(z)/T$ |
| $H$ | **數值積分步長**（⚠️ 不是回授轉移函數！） |

## 📖 本章會用到的 Octave 語法小抄

| 語法 | 意義 | 注意事項 |
|---|---|---|
| `[r,p,k]=residue(n,d)` | 部分分式展開 | `r`=留數、`p`=極點、`k`=直接項 |
| `deconv(a,b)` | 多項式**除法** | 用來把 $(z-1)$ 因式除掉 |
| `polyval(p,x)` | 多項式在 `x` 求值 | 代 $z=1$ |
| `roots(p)` | 多項式的根 | 找 $z=1$ 的極點 |
| `conv(a,b)` | 多項式**乘法** | |
| **`log(x)`** | **自然對數 $\ln x$** | **不是 $\log_{10}$！** |
| `abs(z)`, `angle(z)` | 複數絕對值、輻角 | **`angle` 回傳弧度** |
| `single(x)` | 轉成單精度 | 實驗 7 用來重現捨入誤差 |
| `stairs`, `stem`, `plot` | 三種畫法 | 離散響應用 `stairs` |

> **⚠️ 本章三個致命陷阱**
>
> 1. **`log` 是自然對數**。式 (6-8)~(6-10) 全部用 $\ln$，寫成 `log10` 會完全錯。
> 2. **`angle` 回傳弧度**。式 (6-8)、(6-9) 中的 $\theta$ 必須是弧度；若從書上抄 $51^\circ$ 直接代入，答案會錯——要先乘 $\pi/180$。
> 3. **差分方程式的更新順序**：`cm2=cm1; cm1=c;` 順序不能顛倒，否則會覆蓋掉還要用的值。

---


## 二、一階系統的時間響應 (例 6.1)

### 系統與公式

單位回授，$G_p(s)=\dfrac{4}{s+2}$，$T=0.1$ s。（教材說明：**溫控系統的受控體常被模型化為一階系統**，見第 1.6 節。）

$$G(z)=\mathcal{Z}\left[\frac{1-e^{-Ts}}{s}\cdot\frac{4}{s+2}\right]=\frac{0.3625}{z-0.8187}$$

$$T(z)=\frac{G(z)}{1+G(z)}=\frac{0.3625}{z-0.4562}$$

階躍響應（$R(z)=\dfrac{z}{z-1}$）：

$$C(z)=\frac{0.3625z}{(z-1)(z-0.4562)}=\frac{0.667z}{z-1}+\frac{-0.667z}{z-0.4562}$$

$$\boxed{c(kT)=0.667\left[1-(0.4562)^k\right]}$$

**穩態值 $0.667$，不是 1** —— 有穩態誤差。第 6.5 節會解釋原因（**系統型式 $N=0$**）。

### `residue` 在做什麼

教材的程式用 `residue` 做**部分分式展開**：

```matlab
[r,p,k] = residue([0.3625],[1 -1.4562 0.4562])
```

它把 $\dfrac{C(z)}{z}=\dfrac{0.3625}{z^2-1.4562z+0.4562}$ 展開成

$$\frac{0.6666}{z-1}+\frac{-0.6666}{z-0.4562}$$

回傳三樣：

| 回傳值 | 意義 | 本例 |
|---|---|---|
| `r` | 各項的**留數**（分子） | $[0.6666,\ -0.6666]$ |
| `p` | 各項的**極點**（分母的根） | $[1,\ 0.4562]$ |
| `k` | **直接項**（分子次數 $\ge$ 分母時才有） | 空 `[]` |

> **為什麼要展開 $C(z)/z$ 而不是 $C(z)$？** 因為 z 轉換表的標準形式是 $\dfrac{kz}{z-p}\leftrightarrow kp^n$——**分子有一個 $z$**。先除以 $z$ 展開、再乘回來，就能直接查表。


In [ ]:
%% 實驗 1：一階系統的時間響應（例 6.1）
T  = 0.1;
Gs = tf(4, [1 2]);              % 受控體 Gp(s) = 4/(s+2)
Gz = c2d(Gs, T, 'zoh');         % G(z)
Tz = feedback(Gz, 1);           % 閉迴路

[ng, dg] = tfdata(Gz, 'v');
[nt, dt] = tfdata(Tz, 'v');
printf('G(z) = %.4f / (z %+.4f)    教材 0.3625/(z-0.8187)\n', ng(2), dg(2));
printf('T(z) = %.4f / (z %+.4f)    教材 0.3625/(z-0.4562)\n\n', nt(2), dt(2));

% 教材的部分分式展開
[r, p, kk] = residue([0.3625], [1 -1.4562 0.4562]);
printf('residue 留數 r = [%.4f; %.4f]   教材 [0.6666; -0.6666]\n', r);
printf('residue 極點 p = [%.4f; %.4f]   教材 [1.0000; 0.4562]\n', p);
printf('直接項 k 為空？ %d\n\n', isempty(kk));

% 與教材公式比對
n = 0:8;
[y1, ~] = step(Tz, 0:T:8*T);
c_theory = 0.667*(1 - 0.4562.^n);
printf('  n     模擬 c(nT)    教材公式       差\n');
for i = 1:numel(n)
    printf('  %-5d %-13.6f %-14.6f %.2e\n', n(i), y1(i), c_theory(i), abs(y1(i)-c_theory(i)));
end
printf('\n穩態值 = %.6f  （教材 0.667，注意不是 1 -> 有穩態誤差）\n', dcgain(Tz));

### 結果解讀

模擬與教材公式吻合到 $10^{-4}$ 等級——**這個殘差來自教材把 $0.6666\ldots$ 印成 $0.667$**（第 5 章實驗 6 已經見過同樣的現象）。

**穩態值 $2/3$ 而不是 1**：因為開迴路 $G(z)$ 在 $z=1$ **沒有**極點（**系統型式 $N=0$**），由式 (6-19)：

$$e_{ss}=\frac{1}{1+K_P}=\frac{1}{1+2}=\frac13\quad\Longrightarrow\quad c_{ss}=1-\frac13=\frac23$$

實驗 6 會驗證這個計算。

---


## 三、取樣 vs 不取樣的對照 (例 6.2)

把取樣器與零階保持器**移除**，得到純類比閉迴路：

$$T_a(s)=\frac{G_p(s)}{1+G_p(s)}=\frac{4}{s+6}\quad\Longrightarrow\quad c_a(t)=0.667\left(1-e^{-6t}\right)$$

**兩者穩態值相同（都是 $0.667$），但暫態不同**——這正是取樣造成的影響。


In [ ]:
%% 實驗 2：取樣 vs 不取樣（例 6.2）
Ta = feedback(Gs, 1);           % 拿掉取樣器的純類比閉迴路
[na, da] = tfdata(Ta, 'v');
printf('Ta(s) = %g/(s %+g)    教材 4/(s+6)\n', na(2), da(2));
printf('穩態：類比 %.6f  vs  離散 %.6f  （相同）\n', dcgain(Ta), dcgain(Tz));
printf('時間常數：類比 1/6 = %.4f s  vs  離散 %.4f s\n', 1/6, -T/log(-dt(2)));

figure('Position', [50 50 800 380]);
tt = 0:0.005:0.8;
[ya, ~] = step(Ta, tt);
[yd, td] = step(Tz, 0:T:0.8);
plot(tt, ya, 'b-', 'LineWidth', 1.5); hold on;
stairs(td, yd, 'r-', 'LineWidth', 2); grid on;
title('例 6.1 vs 6.2：取樣資料系統 vs 類比系統');
xlabel('時間 t (秒)'); ylabel('c(t)');
legend('類比系統（無取樣）', '取樣資料系統（T=0.1s）', 'Location', 'southeast');

### 結果解讀

**穩態完全相同，暫態不同。** 離散系統的時間常數 $0.127$ s 比類比的 $0.167$ s **短**——這個例子中取樣反而讓響應變快了一點。

> **注意這不是通則。** 範例 6.6 的二階系統中，取樣的效果是明顯的**去穩定化**（阻尼比從 0.5 掉到 0.25）。**取樣對系統的影響要看具體系統與 $T$ 的大小，不能一概而論。**

---


## 四、差分方程式解法 (例 6.4)

### 從 $T(z)$ 到可執行的遞迴式

**系統**：$G_p(s)=\dfrac{1}{s(s+1)}$，單位回授，$T=1$ s。

$$T(z)=\frac{0.368z+0.264}{z^2-z+0.632}$$

**改寫成 $z^{-1}$ 的形式**（分子分母同除 $z^2$）：

$$\frac{C(z)}{R(z)}=\frac{0.368z^{-1}+0.264z^{-2}}{1-z^{-1}+0.632z^{-2}}$$

**交叉相乘**：

$$C(z)\left[1-z^{-1}+0.632z^{-2}\right]=R(z)\left[0.368z^{-1}+0.264z^{-2}\right]$$

**取反 z 轉換**（$z^{-1}$ 對應延遲一步）：

$$\boxed{c(kT)=0.368\,r(kT-T)+0.264\,r(kT-2T)+c(kT-T)-0.632\,c(kT-2T)}$$

**這是一條可以直接寫成程式的遞迴式**，也就是所有數位濾波器的實作形式。

### 教材的程式與它的核心

```matlab
rm1=0; rm2=0; cm1=0; cm2=0;
for kk=1:14
  k=kk-1;  r=1;
  c = 0.368*rm1 + 0.264*rm2 + cm1 - 0.632*cm2;
  cm2=cm1; cm1=c; rm2=rm1; rm1=r;    % <- 核心在這一行
end
```

> **💡 最後一行就是 $z^{-1}$ 在程式裡的樣子**：把「現在」變成「上一步」、「上一步」變成「上上步」。
>
> **⚠️ 順序不能顛倒**：必須先做 `cm2=cm1` 再做 `cm1=c`。若寫成 `cm1=c; cm2=cm1;`，`cm2` 會拿到剛更新的 `c` 而不是舊的 `cm1`——**整個遞迴就錯了，而且不會報錯**。

| 變數 | 對應 |
|---|---|
| `r` | $r(kT)$ |
| `rm1` | $r(kT-T)$（m = minus） |
| `rm2` | $r(kT-2T)$ |
| `cm1` | $c(kT-T)$ |
| `cm2` | $c(kT-2T)$ |


In [ ]:
%% 實驗 3：差分方程式解法（例 6.4）
T4  = 1;
Gs4 = tf(1, [1 1 0]);           % Gp(s) = 1/(s(s+1))
Gz4 = c2d(Gs4, T4, 'zoh');
Tz4 = feedback(Gz4, 1);
[n4, d4] = tfdata(Tz4, 'v');
printf('T(z) = (%.3f z + %.3f)/(z^2 %+.3f z %+.3f)\n', n4(2), n4(3), d4(2), d4(3));
printf('教材 (0.368z + 0.264)/(z^2 - z + 0.632)\n\n');

% 教材的差分方程式（式 6-4）
rm1 = 0; rm2 = 0; cm1 = 0; cm2 = 0;
c_diff = zeros(1, 14);
for kk2 = 1:14
    r_in = 1;                                        % 單位步階
    c = 0.368*rm1 + 0.264*rm2 + cm1 - 0.632*cm2;     % 式 (6-4)
    c_diff(kk2) = c;
    cm2 = cm1; cm1 = c; rm2 = rm1; rm1 = r_in;       % 延遲一步（順序不能顛倒！）
end

[y4, ~] = step(Tz4, 0:T4:13*T4);
printf('  k     差分方程式     step()        差\n');
for i = 1:14
    printf('  %-5d %-14.6f %-13.6f %.2e\n', i-1, c_diff(i), y4(i), abs(c_diff(i)-y4(i)));
end
printf('\n教材列出：k=1: 0.3680，k=11: 1.0809，k=12: 1.0323，k=13: 0.9812\n');

figure('Position', [50 50 800 380]);
stairs(0:13, c_diff, 'b-', 'LineWidth', 2); hold on;
plot([0 13], [1 1], 'k--'); grid on;
title('例 6.4：差分方程式解出的階躍響應（T = 1 s）');
xlabel('k'); ylabel('c(kT)');
legend('c(kT)（差分方程式）', '目標值 1', 'Location', 'southeast');

### 結果解讀

差分方程式與 `step()` 的結果吻合到 $10^{-4}$ 等級（殘差同樣來自**教材係數的四捨五入**：$0.368$ vs $e^{-1}=0.367879$）。

$k=11,12,13$ 的值 $1.0809,\ 1.0323,\ 0.9812$ **與教材列出的完全相同**。

**響應有明顯超越**（最大約 1.40）並且來回震盪——因為 $T=1$ s 太大，阻尼比只有 0.25（實驗 4 會算）。

> **💡 教材的方法論提醒**：許多範例刻意把取樣頻率選得很低，一是讓手算簡單，二是 $T$ 大時 $C(z)$ 的級數只需幾項就能看出響應特性。
>
> **但若 $T=0.1$ s，要得到 $t=0\sim2$ s 的響應需要 21 項**——因此對這種系統，**唯一實用的方法是模擬**。這也是為什麼實務上不用轉換法算時間響應。

---


## 五、由 $z$ 平面極點反推 $\zeta$、$\omega_n$、$\tau$ (式 6-8 ~ 6-10)

**這是本章最實用的三條公式。**

### 基本對應

$$\boxed{s\text{ 平面極點 }s_1\ \longrightarrow\ z\text{ 平面極點 }z_1=e^{s_1T}}$$

反過來用：**$z$ 平面極點 $z_1$，在取樣瞬間會產生與等效 $s$ 平面極點 $s_1$ 相同的暫態特性**。

### 共軛複數極點的意義（式 6-7）

$z=r\angle(\pm\theta)$ 對應的暫態響應項為：

$$\boxed{A\,r^{k}\cos(\theta k+\eta)}$$

| $z$ 極點的座標 | 決定響應的什麼 |
|---|---|
| **$r$（絕對值）** | **衰減速度**：每步乘 $r$。$r<1$ 才衰減 |
| **$\theta$（輻角）** | **振盪速度**：每步相位前進 $\theta$ 弧度 |

### 三條反推公式

由 $r=e^{-\zeta\omega_nT}$ 與 $\theta=\omega_nT\sqrt{1-\zeta^2}$ 解出：

$$\zeta=\frac{-\ln r}{\sqrt{(\ln r)^2+\theta^2}}\qquad\text{(6-8)}$$

$$\omega_n=\frac{1}{T}\sqrt{(\ln r)^2+\theta^2}\qquad\text{(6-9)}$$

$$\tau=\frac{1}{\zeta\omega_n}=\frac{-T}{\ln r}\qquad\text{(6-10)}$$

### 取樣率夠不夠？（式 6-11、6-12）

$$\frac{\tau}{T}=\frac{-1}{\ln r}\quad\text{（每時間常數取樣幾次）},\qquad
\frac{T_d}{T}=\frac{360^\circ}{\theta^\circ}\quad\text{（每振盪週期取樣幾次）}$$

**Table 6-2 摘要**：

| $r$ | $\tau/T$ | | $\theta$ | $T_d/T$ |
|---|---|---|---|---|
| 0.99 | 99.5 | | $20^\circ$ | 18 |
| 0.9 | 9.5 | | $45^\circ$ | 8 |
| 0.8 | 4.48 | | $60^\circ$ | 6 |
| 0.6 | 1.96 | | $120^\circ$ | 3 |
| 0.4 | 1.09 | | $180^\circ$ | 2 |

> **⚠️ 程式實作的兩個陷阱**
> 1. **`log` 是自然對數 $\ln$**，不是 $\log_{10}$。
> 2. **`angle` 回傳弧度**——式 (6-8)、(6-9) 的 $\theta$ 必須是弧度。從書上抄 $51^\circ$ 直接代會錯。


In [ ]:
%% 實驗 4：由 z 極點反推 zeta, wn, tau（式 6-8~6-10，例 6.5、6.6）

% ---- 例 6.5：一階系統的實極點 ----
z1 = -dt(2);                              % = 0.4562
s1 = log(z1)/T;                           % log 是自然對數！
printf('[例 6.5] z1 = %.4f -> s1 = ln(z1)/T = %.4f   教材 -7.848\n', z1, s1);
printf('         tau = 1/|s1| = %.4f s   教材 0.127\n', 1/abs(s1));
printf('         約 4 個時間常數安定 -> %.2f s\n\n', 4/abs(s1));

% ---- 例 6.6：二階系統的共軛複數極點 ----
p4   = pole(Tz4);
rr   = abs(p4(1));
th   = abs(angle(p4(1)));                 % angle 回傳「弧度」
zeta = -log(rr)/sqrt(log(rr)^2 + th^2);   % 式 (6-8)
wn   = sqrt(log(rr)^2 + th^2)/T4;         % 式 (6-9)
tau  = -T4/log(rr);                       % 式 (6-10)

printf('[例 6.6] 極點 = %.4f +- j%.4f = %.4f angle +-%.2f deg (= %.4f rad)\n', ...
       real(p4(1)), abs(imag(p4(1))), rr, th*180/pi, th);
printf('         教材 0.5 +- j0.618 = 0.795 angle 51.0 deg = 0.890 rad\n');
printf('         zeta = %.4f   教材 0.250\n', zeta);
printf('         wn   = %.4f   教材 0.9191 rad/s\n', wn);
printf('         tau  = %.4f   教材 4.36 s\n\n', tau);

% 對照連續系統
pa = pole(feedback(Gs4, 1));
printf('         對照連續系統：zeta = %.4f, wn = %.4f, tau = %.4f （教材 0.5, 1, 2）\n', ...
       -real(pa(1))/abs(pa(1)), abs(pa(1)), 1/abs(real(pa(1))));
printf('         => 取樣的效果是「去穩定化」：阻尼比從 0.5 掉到 0.25\n\n');

% T = 0.1 時取樣影響很小
Tz5 = feedback(c2d(Gs4, 0.1, 'zoh'), 1);
p5  = pole(Tz5); r5 = abs(p5(1)); th5 = abs(angle(p5(1)));
printf('         若改成 T=0.1：zeta = %.4f, wn = %.4f, tau = %.4f\n', ...
       -log(r5)/sqrt(log(r5)^2+th5^2), sqrt(log(r5)^2+th5^2)/0.1, -0.1/log(r5));
printf('         教材 0.475, 0.998, 2.11 -> 幾乎回到連續系統\n\n');

% 取樣率檢查（式 6-11、6-12）
printf('取樣率檢查（式 6-11、6-12）：\n');
printf('  每個時間常數取樣 tau/T = -1/ln(r) = %.2f 次\n', -1/log(rr));
printf('  每個振盪週期取樣 Td/T = 360/theta_deg = %.2f 次\n', 360/(th*180/pi));
printf('  Table 6-2：r=0.8 -> 4.48 次；theta=60deg -> 6 次\n');
printf('  => 本例每週期只取樣 7 次，偏少，這正是阻尼被破壞的原因\n');

### 結果解讀

**三個參數與教材完全吻合**（$\zeta=0.2494$ vs $0.250$、$\omega_n=0.9197$ vs $0.9191$、$\tau=4.36$）。

**取樣的效果是去穩定化**：

| | 連續系統 | 離散（$T=1$） | 離散（$T=0.1$） |
|---|---|---|---|
| $\zeta$ | 0.500 | **0.249** | 0.475 |
| $\omega_n$ | 1.000 | 0.920 | 0.999 |
| $\tau$ | 2.00 | **4.36** | 2.11 |

**$T=1$ 時阻尼比腰斬、時間常數翻倍**；$T=0.1$ 時幾乎回到連續系統。

> **這印證了第 3 章的結論**：ZOH 等效引入 $T/2$ 的延遲，$T$ 越大相位落後越嚴重、阻尼被吃掉越多。
>
> **取樣率檢查也說明了同一件事**：$T=1$ 時每個振盪週期只取樣 7 次（Table 6-2 建議 $\theta\le60^\circ$，也就是至少 6 次；這裡剛好在邊緣），每個時間常數只取樣 4.36 次——**都偏少**。

---


## 六、$s$ 平面映射到 $z$ 平面的三種軌跡

| $s$ 平面軌跡 | $z$ 平面 | 教材圖 |
|---|---|---|
| **等阻尼**（$\sigma$ 常數的垂直線） | **圓**（半徑 $e^{\sigma T}$） | Fig. 6-7 |
| **等頻率**（$\omega$ 常數的水平線） | **射線**（輻角 $\omega T$） | Fig. 6-8 |
| **等阻尼比**（$\omega/\sigma=\tan\delta$ 的斜線） | **對數螺線** | Fig. 6-9 |

**最重要的一條**：

$$\boxed{s\text{ 平面的左半平面（穩定）}\ \longleftrightarrow\ z\text{ 平面的單位圓內（穩定）}}$$

沿 $j\omega$ 軸（$\sigma=0$）：$z=e^{j\omega T}=1\angle\omega T$ ——**恰好是單位圓**。

> **$z$ 平面極點落在單位圓上 = 系統自然響應含持續振盪**，振盪頻率 = 極點輻角（弧度）$/T$。

下一格把這三種軌跡畫出來。**畫法**：在 $s$ 平面上取一系列點，用 `exp(s*T)` 映射到 $z$ 平面即可。


In [ ]:
%% 實驗 5：s 平面映射到 z 平面（Fig. 6-6 ~ 6-9）
th_c = linspace(0, 2*pi, 400);
Tm   = 1;

figure('Position', [50 50 950 440]);

subplot(1,2,1);   % 等阻尼（sigma 常數）-> 圓
plot(cos(th_c), sin(th_c), 'k--', 'LineWidth', 1.2); hold on;
for sig = [-0.1 -0.3 -0.7 -1.5]
    rr2 = exp(sig*Tm);                    % z = e^{sigma*T}
    plot(rr2*cos(th_c), rr2*sin(th_c), 'LineWidth', 1.5);
end
axis equal; grid on;
title('等阻尼軌跡（\sigma 常數）-> 圓');
xlabel('Re(z)'); ylabel('Im(z)');
legend('單位圓（\sigma=0）', '\sigma=-0.1', '\sigma=-0.3', '\sigma=-0.7', '\sigma=-1.5', ...
       'Location', 'southoutside');

subplot(1,2,2);   % 等阻尼比（zeta 常數）-> 對數螺線
plot(cos(th_c), sin(th_c), 'k--', 'LineWidth', 1.2); hold on;
for zt = [0.1 0.3 0.5 0.7]
    wn_v = linspace(0.01, pi/Tm/sqrt(1-zt^2), 300);
    s_v  = -zt*wn_v + 1j*wn_v.*sqrt(1-zt^2);   % s = -zeta*wn + j*wn*sqrt(1-zeta^2)
    z_v  = exp(s_v*Tm);                         % z = e^{sT}
    plot(real(z_v), imag(z_v), 'LineWidth', 1.5);
    plot(real(z_v), -imag(z_v), 'LineWidth', 1.5, 'HandleVisibility', 'off');
end
axis equal; grid on;
title('等阻尼比軌跡（\zeta 常數）-> 對數螺線');
xlabel('Re(z)'); ylabel('Im(z)');
legend('單位圓', '\zeta=0.1', '\zeta=0.3', '\zeta=0.5', '\zeta=0.7', ...
       'Location', 'southoutside');

printf('左圖：sigma 越負 -> 圓越小 -> 極點越靠近原點 -> 衰減越快\n');
printf('右圖：這就是控制教科書常見的「z 平面等阻尼比格線」\n');
printf('      設計時把極點放在某條螺線上，就等於指定了阻尼比\n');

### 結果解讀

**左圖（等阻尼）**：$\sigma$ 越負，圓的半徑 $e^{\sigma T}$ 越小——**極點越靠近原點，衰減越快**。單位圓（$\sigma=0$）是穩定邊界。

**右圖（等阻尼比）**：這就是控制教科書常見的 **$z$ 平面設計格線**。設計時把閉迴路極點放在某條螺線上，就等於**指定了阻尼比**（也就等於指定了超越量）。

> **這兩張圖是第 8、9 章控制器設計的基礎工具**：規格說「超越量小於 20%」→ 查出對應的 $\zeta$ → 在 $z$ 平面上找到對應的螺線 → 把閉迴路極點放上去。**這正是 [`course/ackermann`](../ackermann/full-Ackermann-formula-example.md) 中「向系統下訂單」那一步在做的事。**

---


## 七、穩態精確度：系統型式 (6.5)

### 系統型式 $N$

$$G(z)=\frac{K\prod_{i=1}^{m}(z-z_i)}{(z-1)^{N}\prod_{j=1}^{p}(z-z_j)},\qquad z_i\neq1,\ z_j\neq1$$

$$\boxed{N=G(z)\text{ 在 }z=1\text{ 的極點數，稱為「系統型式」}}$$

### 兩個誤差常數與穩態誤差

$$K_P=\lim_{z\to1}G(z)\qquad\text{（位置誤差常數，式 6-18）}$$

$$K_v=\lim_{z\to1}\frac{1}{T}(z-1)G(z)\qquad\text{（速度誤差常數，式 6-20）}$$

| 系統型式 | 步階輸入 $e_{ss}$ | 斜坡輸入 $e_{ss}$ |
|---|---|---|
| $N=0$ | $\dfrac{1}{1+K_P}$ | $\infty$ |
| $N=1$ | $0$ | $\dfrac{1}{K_v}=\dfrac{T}{K_{dc}}$ |
| $N\ge2$ | $0$ | $0$ |

### 💡 設計上的取捨（教材的重要結論）

> **提高增益、或在開迴路增加 $z=1$ 的極點，都會減少穩態誤差。**
>
> **但（第 7 章會證明）大增益與 $z=1$ 的極點，兩者都有「去穩定化」效果。**
>
> **因此「小穩態誤差」與「足夠穩定度」之間存在取捨。**

### 程式怎麼算 $K_v$

$K_v=\lim_{z\to1}\dfrac{(z-1)G(z)}{T}$ 直接代 $z=1$ 會得到 $\dfrac{0}{0}$。**解法**：分母含因式 $(z-1)$，先用 `deconv` 把它除掉，剩下的部分再代 $z=1$。

```matlab
d_rest = deconv(den, [1 -1]);          % 多項式除法，把 (z-1) 除掉
Kv = polyval(num,1) / polyval(d_rest,1) / T;
```

| 函數 | 作用 |
|---|---|
| `deconv(a, b)` | 多項式**除法**（`conv` 的反運算） |
| `polyval(p, 1)` | 把 $z=1$ 代進多項式 |
| `roots(d)` | 找分母的根，用來數 $z=1$ 的極點個數 |


In [ ]:
%% 實驗 6：穩態精確度、系統型式、Kp、Kv（式 6-18~6-21，例 6.7）
T6 = 0.1;

% ---- 型式 0：G(z) 在 z=1 沒有極點 ----
G0  = c2d(tf(4, [1 2]), T6, 'zoh');
[n0g, d0g] = tfdata(G0, 'v');       % 加了 'v' 回傳的是矩陣，不能用 {} 索引
Kp0 = dcgain(G0);
printf('[型式 0] Gp(s) = 4/(s+2)\n');
printf('   N  = %d\n', sum(abs(roots(d0g) - 1) < 1e-8));
printf('   Kp = lim G(z) = %.4f\n', Kp0);
printf('   步階 ess = 1/(1+Kp) = %.4f\n', 1/(1+Kp0));
printf('   實測 1 - dcgain(閉迴路) = %.4f  <- 吻合\n', 1 - dcgain(feedback(G0,1)));
printf('   斜坡 ess = 無限大（Kv = 0）\n\n');

% ---- 型式 1：G(z) 在 z=1 有一個極點（例 6.7，K=1）----
K  = 1;
G1 = c2d(tf(K, [1 1 0]), T6, 'zoh');       % K/(s(s+1))
[n1g, d1g] = tfdata(G1, 'v');
% Kv = lim (z-1)G(z)/T：分母含 (z-1)，先用 deconv 除掉再代 z=1，避免 0/0
d_rest = deconv(d1g, [1 -1]);
Kv1 = polyval(n1g, 1) / polyval(d_rest, 1) / T6;

printf('[型式 1] Gp(s) = K/(s(s+1))，K = %g（例 6.7）\n', K);
printf('   N  = %d\n', sum(abs(roots(d1g) - 1) < 1e-8));
printf('   Kv = lim (z-1)G(z)/T = %.4f   教材說 Kv = K = %g\n', Kv1, K);
printf('   步階 ess = 0（因為 N >= 1）\n');
printf('   實測 1 - dcgain(閉迴路) = %.2e  <- 確實為 0\n', 1 - dcgain(feedback(G1,1)));
printf('   斜坡 ess = 1/Kv = %.4f\n\n', 1/Kv1);

% ---- 增益 K 對穩態誤差的影響 ----
printf('增益 K 對斜坡穩態誤差的影響（型式 1）：\n');
printf('  K       Kv        斜坡 ess = 1/Kv\n');
for Kx = [0.5 1 2 5 10]
    Gx = c2d(tf(Kx, [1 1 0]), T6, 'zoh');
    [nx, dx] = tfdata(Gx, 'v');
    Kvx = polyval(nx,1) / polyval(deconv(dx,[1 -1]),1) / T6;
    printf('  %-7g %-9.4f %.4f\n', Kx, Kvx, 1/Kvx);
end
printf('  => 增益越大穩態誤差越小，但第 7 章會證明大增益會讓系統不穩定\n');

### 結果解讀

**型式 0**：$K_P=2$，步階穩態誤差 $=\dfrac{1}{1+2}=\dfrac13=0.3333$——**與實驗 1 的 $c_{ss}=0.667$ 完全對應**（$1-0.3333=0.6667$）。

**型式 1**：步階誤差為 0（實測 $10^{-15}$ 等級），斜坡誤差 $=\dfrac{1}{K_v}=\dfrac1K$——**與教材例 6.7 的結論一致**。

**增益的效果**：$K$ 從 0.5 增到 10，斜坡誤差從 2 降到 0.1。

> **但這正是教材警告的取捨**：第 7 章會證明**大增益會讓系統不穩定**。而且——
>
> **⚠️ 誤差分析在系統穩定性未獲保證之前是沒有意義的**（教材原話）。$e_{ss}$ 是用終值定理算的，而終值定理**要求終值存在**——系統不穩定時根本沒有終值。

### 例 6.8：用 PI 補償器把型式 0 變成型式 1

若受控體沒有 $z=1$ 的極點（型式 0），對斜坡輸入的誤差是無限大。**解法**是加一個 **PI 補償器**：

$$D(z)=\frac{K_Iz}{z-1}+K_P$$

它**自帶一個 $z=1$ 的極點**，把系統從型式 0 升級成型式 1。代入式 (6-20) 可得 $K_v=\dfrac{K_I}{T}$，因此要達成 $e_{ss}<0.01$ 需要：

$$\boxed{K_I=100T}$$

**這就是第 8 章 PID 控制器設計的起點。**

---


## 八、模擬與 Euler 法 (6.6)

### 為什麼需要模擬

第 6.2 節的結論：**對高階系統，模擬是唯一實用的技術**。轉換法（拉氏／z 轉換）**只適用於線性系統**，模擬則不受此限。

### 矩形法則（Euler 法，式 6-23）

要數值積分 $x(t)=\displaystyle\int_0^t y(\nu)d\nu+x(0)$。假設 $y(t)$ 在 $(k-1)H\le t<kH$ 期間**為常數**：

$$\boxed{x(kH)=x[(k-1)H]+H\,y[(k-1)H]}$$

**幾何意義**：用**矩形面積**近似曲線下的面積。$H$ 稱為**積分步長**。

### 教材的範例

$\dot{x}+x=0$，$x(0)=1$，精確解 $x(t)=e^{-t}$。因為 $y=\dot{x}=-x$：

$$x(kH)=x[(k-1)H]\left(1-H\right)$$

取 $H=0.1$：$x(1.0)=0.9^{10}=0.3487$，而精確值 $e^{-1}=0.3678$——**誤差約 5.2%**。

### ⚠️ 步長的取捨（教材的重要警告）

> **$H$ 取大 → 誤差大。**
> **$H$ 減小 → 誤差先減到一個最小值。**
> **$H$ 再減小 → 誤差反而增加**，因為電腦的**捨入誤差**。
>
> 用 $H=0.1$ 算 $x(1)$ 需要 10 次迭代；$H=0.001$ 需要 1000 次。**後者的捨入誤差較大，因為做了更多次計算。**

**存在一個「最佳步長」。**

> **💡 本 Notebook 的補充**：在**雙精度**下，即使 $H=10^{-7}$（一千萬次迭代）誤差仍在下降——雙精度太準，看不到反轉。**改用單精度 `single` 就能清楚重現教材描述的現象**。下一格會同時跑兩種精度做對照。


In [ ]:
%% 實驗 7：Euler 法與步長的取捨（式 6-23）
printf('解 xdot + x = 0，x(0) = 1，精確解 x(t) = e^{-t}\n\n');
printf('  H        步數     Euler x(1.0)    誤差\n');
for H = [0.5 0.2 0.1 0.05 0.01]
    N = round(1/H); x = 1;
    for i = 1:N, x = x + H*(-x); end            % 式 (6-23)
    printf('  %-8g %-8d %-15.8f %.2e\n', H, N, x, abs(x - exp(-1)));
end
printf('  精確值 e^{-1} = %.8f\n', exp(-1));
printf('  教材：H=0.1 時 x(1.0) = 0.3487，精確 0.3678\n\n');

% 教材警告的「H 過小時誤差反而增加」：雙精度太準看不到，用單精度重現
printf('教材警告「H 過小時捨入誤差反而增加」的實測：\n');
printf('  H          步數         single 誤差     double 誤差\n');
for H = [1e-2 1e-3 1e-4 1e-5 1e-6 1e-7]
    N = round(1/H);
    xs = single(1); Hs = single(H);
    for i = 1:N, xs = xs + Hs*(-xs); end        % 單精度
    xd = 1;
    for i = 1:N, xd = xd + H*(-xd); end         % 雙精度
    printf('  %-10.0e %-12d %-15.3e %.3e\n', H, N, abs(double(xs)-exp(-1)), abs(xd-exp(-1)));
end

figure('Position', [50 50 800 380]);
H = 0.1; N = 20; xe = zeros(1,N+1); xe(1) = 1;
for i = 1:N, xe(i+1) = xe(i) + H*(-xe(i)); end
plot(0:0.01:2, exp(-(0:0.01:2)), 'b-', 'LineWidth', 1.5); hold on;
plot((0:N)*H, xe, 'ro--', 'LineWidth', 1.5, 'MarkerFaceColor', 'r'); grid on;
title('Euler 法（矩形法則）vs 精確解，H = 0.1');
xlabel('時間 t (秒)'); ylabel('x(t)');
legend('精確解 e^{-t}', 'Euler 法', 'Location', 'northeast');

### 結果解讀

**第一張表**：$H=0.1$ 時 $x(1.0)=0.34867844$，與教材的 $0.3487$ 完全吻合。誤差隨 $H$ 減小而**單調下降**。

**第二張表才是重點**——它重現了教材的警告：

| $H$ | 單精度誤差 | 雙精度誤差 |
|---|---|---|
| $10^{-4}$ | $1.9\times10^{-5}$ | $1.8\times10^{-5}$ |
| $10^{-5}$ | $\mathbf{4.3\times10^{-7}}$ ← 最小 | $1.8\times10^{-6}$ |
| $10^{-6}$ | $2.6\times10^{-5}$ ← **反轉惡化** | $1.8\times10^{-7}$ |
| $10^{-7}$ | $3.1\times10^{-2}$ ← **災難** | $1.8\times10^{-8}$ |

**單精度在 $H\approx10^{-5}$ 時誤差最小**，再減小 $H$ 反而急速惡化——$H=10^{-7}$ 時誤差比 $H=10^{-2}$ 還大 17 倍。

**背後的機制**：

```text
截斷誤差（truncation）：隨 H 減小而「下降」  ← 矩形近似不夠精確
捨入誤差（round-off）  ：隨迭代次數「上升」   ← 每次加法都損失精度
                         ↓
                    兩者有一個最佳交點
```

**雙精度**因為有約 16 位有效數字，捨入誤差要到極小的 $H$ 才顯現，所以第一張表看不出來。**教材寫作年代的運算精度較低，這個現象更明顯。**

> **💡 實務結論**：不要以為步長取越小越準。**每個數值積分演算法都有一個最佳步長**，而且更好的做法是改用高階演算法（Runge-Kutta 等），它們在同樣步長下截斷誤差小得多。

---


## 九、本章總結

### 公式速查表

| 式號 | 公式 | 用途 | 對應實驗 |
|---|---|---|---|
| (6-4) | 差分方程式遞迴 | 解時間響應 | 3 |
| — | $1+\overline{GH}(z)=0$ | **系統特徵方程式** | — |
| — | $z_1=e^{s_1T}$ | $s$ 平面 → $z$ 平面 | 4, 5 |
| (6-7) | $z=r\angle\pm\theta\to Ar^k\cos(\theta k+\eta)$ | $z$ 極點 → 響應形式 | 4 |
| **(6-8)** | $\zeta=\dfrac{-\ln r}{\sqrt{(\ln r)^2+\theta^2}}$ | **反推阻尼比** | 4 |
| **(6-9)** | $\omega_n=\dfrac{1}{T}\sqrt{(\ln r)^2+\theta^2}$ | **反推自然頻率** | 4 |
| **(6-10)** | $\tau=\dfrac{-T}{\ln r}$ | **反推時間常數** | 4 |
| (6-11)(6-12) | $\dfrac{\tau}{T}=\dfrac{-1}{\ln r}$，$\dfrac{T_d}{T}=\dfrac{360^\circ}{\theta^\circ}$ | 取樣率夠不夠 | 4 |
| (6-18) | $K_P=\lim_{z\to1}G(z)$ | 位置誤差常數 | 6 |
| (6-20) | $K_v=\lim_{z\to1}\dfrac{(z-1)G(z)}{T}$ | 速度誤差常數 | 6 |
| (6-23) | $x(kH)=x[(k-1)H]+Hy[(k-1)H]$ | **Euler 法** | 7 |

### MATLAB / Octave 常見錯誤

| 錯誤 | 症狀 | 正確做法 |
|---|---|---|
| **用 `log10` 代替 `log`** | $\zeta$、$\omega_n$、$\tau$ 全錯 | **`log` 就是 $\ln$** |
| **把 $\theta$ 用度數代入式 (6-8)** | 結果錯很多 | `angle()` 回傳弧度，直接用 |
| 差分方程式更新順序顛倒 | 遞迴錯誤，**不會報錯** | `cm2=cm1;` 一定在 `cm1=c;` 之前 |
| 直接代 $z=1$ 算 $K_v$ | 得到 `0/0` = `NaN` | 先 `deconv(den,[1 -1])` 除掉因式 |
| `fprintf('%.5f', pole(sys))` | 複數極點的虛部被吃掉 | 用 `[real(p)'; imag(p)']` |
| 以為步長越小越準 | 捨入誤差反而放大 | 有最佳步長；或改用高階演算法 |
| 抄教科書四捨五入過的係數 | 結果差 $10^{-4}$ 等級 | 讓程式自己算 |

### 承先啟後

本章建立了**把 $z$ 平面極點翻譯成物理行為**的完整語言：

$$z=r\angle\pm\theta\quad\Longrightarrow\quad \zeta,\ \omega_n,\ \tau,\ \text{超越量},\ \text{取樣率是否足夠}$$

同時也建立了**穩態誤差的判準**（系統型式 $N$、$K_P$、$K_v$）。

**第 7 章**用這些工具討論**穩定度**——極點是否都在單位圓內，以及 Routh-Hurwitz、Jury、根軌跡、Nyquist、Bode 等判別法。

**第 8、9 章**則是**設計** $D(z)$，讓閉迴路極點落到你指定的位置——那正是 [`course/ackermann`](../ackermann/full-Ackermann-formula-example.md) 示範的極點安置法。本章實驗 5 畫的等阻尼比螺線，就是那時候用來「下訂單」的地圖。
